# étoiles Be 
$\rightarrow$ **analyse de l'étoile Be KX And**
- vitesse de rotation du gaz
- vitesse de rotation de l'étoile
- taille du disque du gaz

A faire : 
- v558
- gam Cas
- 20240826_hd183656_v923aql_altair_tcrb
- 20250822_v1296Aql_altair_Tcrb_rsOph (Be herbig)



## spectro dashboard
- lancer la cellule suivante
- sur 'Colormap', bouton droit : "create new view for cell output"
- redimensionner ou déplacer l'onglet créé 'Output View' 

In [11]:
%matplotlib widget
import numpy as np
from spectro_dashboard import SpectroDashboard

# 1. Afficher le dashboard
db = SpectroDashboard()
db.show()


# double ajustement des deux ailes de H alpha

In [12]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.modeling import models, fitting
from specutils import Spectrum1D
from astropy import units as u

# c
filename = 'data/plouis/_kxand_20251125_993.fits' # <-- Vérifiez le nom du fichier

# Lecture
sp = Spectrum1D.read(filename)

# Extraction des valeurs brutes (pour éviter les soucis avec les unités astropy)
x_data = sp.spectral_axis.value
y_data = sp.flux.value 

# Découpage de la zone H-alpha (6540 - 6590 A)
mask = (x_data > 6540) & (x_data < 6590)
x_window = x_data[mask]
y_window = y_data[mask]

# Initialisation pour aider les modèles
y_max = np.max(y_window)
y_min = np.min(y_window)

# modele sur la raie de gauche
g_init_V = models.Gaussian1D(amplitude=y_max, mean=6560, stddev=2.0)

# modele sur la raie de droite
g_init_R = models.Gaussian1D(amplitude=y_max, mean=6566, stddev=2.0)

# on prépare le continuum 
continuum_init = models.Const1D(amplitude=y_min)

model_init = g_init_V + g_init_R + continuum_init

# On ajuste sur les données fenêtrées
fitter = fitting.LevMarLSQFitter()
fit = fitter(model_init, x_window, y_window)

# fit[0] est la Gaussienne V, fit[1] est la Gaussienne R
center_V = fit[0].mean.value
center_R = fit[1].mean.value

# Calcul de la vitesse
delta_lambda = abs(center_R - center_V)
lambda_0 = 6562.8 # H-alpha repos
c = 299792.458
v_rot_disk = (c * (delta_lambda / lambda_0)) / 2

# on affiche le tout
db.load_spectrum(x_window, y_window, label='Spectre Brut')
db.load_spectrum(x_window, fit(x_window), label='Fit Double Gaussienne')

# Les Lignes Verticales (Centres des Gaussiennes)
l_v = db.ax_spec.axvline(center_V, color='blue', linestyle='-', alpha=0.8, label=f'Centre V ({center_V:.2f} A)')
l_r = db.ax_spec.axvline(center_R, color='green', linestyle='-', alpha=0.8, label=f'Centre R ({center_R:.2f} A)')

# et les résultats
print(f"Position du Pic Bleu (Modélisé)  : {center_V:.3f} A")
print(f"Position du Pic Rouge (Modélisé) : {center_R:.3f} A")
print(f"Séparation (Delta Lambda)        : {delta_lambda:.3f} A")
print("-" * 40)
print(f"VITESSE DE ROTATION DU DISQUE    : {v_rot_disk:.1f} km/s")
print("-" * 40)

Spectre Brut : dispersion=0.0309 Å/px
Fit Double Gaussienne : dispersion=0.0309 Å/px
Position du Pic Bleu (Modélisé)  : 6560.339 A
Position du Pic Rouge (Modélisé) : 6564.529 A
Séparation (Delta Lambda)        : 4.190 A
----------------------------------------
VITESSE DE ROTATION DU DISQUE    : 95.7 km/s
----------------------------------------


# ajustement de la raie HeI

In [13]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.modeling import models, fitting
from specutils import Spectrum1D
from astropy import units as u

R_power = 19000 
lambda_HeI = 6678.15 # Longueur d'onde au repose de He I

# Calcul de la largeur instrumentale : FWHM_inst = Lambda / R
fwhm_inst = lambda_HeI / R_power

# Chargement
sp = Spectrum1D.read(filename)
x_data = sp.spectral_axis.value
y_data = sp.flux.value

# on encadre heI
mask = (x_data > 6660) & (x_data < 6700)
x_window = x_data[mask]
y_window = y_data[mask]

# On cherche un continuum (constante) + une Gaussienne NÉGATIVE (trou)
y_cont_estim = np.max(y_window) # Le continuum est le haut du signal
y_dip_estim = np.min(y_window) - y_cont_estim # La profondeur du trou

# on initialize le modèle : Gaussienne (Absorption) + Constante
g_abs = models.Gaussian1D(amplitude=y_dip_estim, mean=lambda_HeI, stddev=2.0)
continuum = models.Const1D(amplitude=y_cont_estim)
model_he = g_abs + continuum

# on ajuste
fitter = fitting.LevMarLSQFitter()
fit = fitter(model_he, x_window, y_window)

# Largeur mesurée (attention ici la FWHM = vsini)
stddev_mesure = fit[0].stddev.value

# inceritudes 
fwhm_mesure = 2.355 * stddev_mesure

# Correction de la résolution instrumentale (moyenne quadratique)
# On enlève le flou du spectro pour avoir la vraie largeur de la raie
if fwhm_mesure > fwhm_inst:
    fwhm_star = np.sqrt(fwhm_mesure**2 - fwhm_inst**2)
else:
    fwhm_star = 0 # Cas impossible (étoile plus fine que le spectro !)

# Conversion en vitesse (Formule Doppler)
# Le facteur 1.2-1.3 compense l'assombrissement centre-bord (Limb Darkening)
coeff_ld = 1.3 
c = 299792.458
v_sin_i = (c * fwhm_star) / (lambda_HeI * coeff_ld)


# on affiche le tout
db.clear_spectra()
#l_r.remove()
#l_v.remove()

# spectre brut
db.load_spectrum(x_window, y_window, label='spectre')

# le Fit
db.load_spectrum(x_window, fit(x_window), label='Fit')

# et les résultats
print(f"FWHM Instrumentale (R={R_power}) : {fwhm_inst:.3f} A")
print(f"FWHM Mesurée (Totale)        : {fwhm_mesure:.3f} A")
print(f"FWHM Étoile (Corrigée)       : {fwhm_star:.3f} A")
print("-" * 40)
print(f"VITESSE DE ROTATION (v sin i) : {v_sin_i:.0f} km/s")
print("-" * 40)

spectre : dispersion=0.0309 Å/px
Fit : dispersion=0.0309 Å/px
FWHM Instrumentale (R=19000) : 0.351 A
FWHM Mesurée (Totale)        : 7.401 A
FWHM Étoile (Corrigée)       : 7.393 A
----------------------------------------
VITESSE DE ROTATION (v sin i) : 255 km/s
----------------------------------------


## analyse

KX And est une toupie binaire rapide ($v \sin i \approx 255$ km/s) avec un disque dense et étendu ($V_{disk} \approx 95$ km/s).

- La force centrifuge est si violente à l'équateur qu'elle contre la gravité.
- L'étoile est un sphéroïde aplati (oblate). Elle est beaucoup plus large à l'équateur qu'aux pôles.
- Comme l'équateur est plus "loin" du centre, la gravité y est plus faible : c'est ce qui permet au gaz de s'échapper facilement pour former le disque.

Pour exemples: 
- soleil (G2V) ~2 km/s
- une B Normale (ex: Vega) ~20 km/s (pôle) à 100 km/s (équateur)

<img src="be_stars.jpg" alt="étoiles Be" width="600" height="500">

Attention au $v \sin i \approx 255$ km/s) :
- Si $i \approx 0^\circ$ (Vu du dessus / Pôle) : Pas d'effet Doppler différentiel -> pic d'émission unique et fin.
- Si $i \approx 45^\circ$ : -> double pic, mais sans absorption centrale marquée.
- Si $i \approx 90^\circ$ (Vu par la tranche / Équateur) : on voit à travers le disque. Le gaz froid du disque passe devant l'étoile et bloque sa lumière -> creux d'absorption caractéristique.


il y a quelques millions d'années :
- Le système était composé de deux étoiles normales.
- L'étoile compagne (celle qui est invisible aujourd'hui ou très faible) était plus massive et a évolué plus vite. Elle a gonflé.
- En gonflant, elle a déversé sa matière sur l'étoile B (notre KX And).
- La matière ne tombe pas tout droit : elle arrive en spiralant, chargée d'énergie cinétique.


Si on applique la loi de Kepler ($V \propto 1/\sqrt{R}$), on peut estimer la taille moyenne de l'anneau d'émission H-alpha par rapport à l'étoile :$$R_{disque} \approx \left( \frac{V_{étoile}}{V_{disque}} \right)^2 \approx \left( \frac{291}{94} \right)^2 \approx \mathbf{9.6}$$
